# Explorando Dados de Saúde Mental – Competição Kaggle
Este notebook tem como objetivo resolver a competição *Explorando Dados de Saúde Mental* do Kaggle, onde precisamos prever a coluna **Depressão** a partir de dados de uma pesquisa de saúde mental.
Os dados de treinamento estão em `train.csv` e os de teste em `test.csv`. A métrica de avaliação é a **Pontuação de Acurácia** e o arquivo de submissão deve conter as colunas `id,Depressão`.
Ao longo deste notebook, faremos: 1) exploração e limpeza dos dados, 2) engenharia de features, 3) validação com holdout, 4) treinamento de modelos (XGBoost, CatBoost, etc.) e 5) geração do arquivo `submission.csv` com as previsões iniciais.
O código foi escrito para rodar em um ambiente Ubuntu 22.04 com 128 GiB de RAM, 16 threads de CPU e sem GPU, respeitando o limite de 60 minutos de execução.


## Etapa 1 – Carregamento dos Dados

Até agora, apenas introduzimos o objetivo do notebook e descrevemos o fluxo geral da competição. O próximo passo é **carregar** os arquivos de dados que serão usados em todas as etapas subsequentes.

### O que faremos nesta célula
- Ler `train.csv` e `test.csv` com `pandas.read_csv`.
- Armazenar os DataFrames em variáveis `df_train` e `df_test`.
- Verificar rapidamente o número de linhas e colunas, bem como a presença de valores ausentes.
- Garantir que a coluna alvo **Depressão** esteja presente apenas no conjunto de treino.

### Por que isso é importante
Carregar os dados corretamente é a base de todo o pipeline. Se houver erros de leitura (por exemplo, separadores diferentes ou codificação incorreta), todas as etapas seguintes falharão ou produzirão resultados inválidos. Além disso, ao inspecionar os dados logo no início, podemos identificar rapidamente problemas como valores ausentes ou tipos de dados inesperados que precisarão ser tratados nas próximas células.

### Próximos passos
Após o carregamento, avançaremos para a **Análise Exploratória dos Dados (EDA)**, onde examinaremos tipos de colunas, distribuição da variável alvo, e identificaremos outliers ou ruído. Em seguida, procederemos ao pré‑processamento, codificação de variáveis categóricas, normalização de numéricos, e criação de features adicionais.

Esta célula não contém código Python, apenas a descrição do que será feito. O código real será inserido na próxima célula de código.


In [ ]:
import pandas as pd

# Carregar os arquivos de dados
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

# Exibir dimensões dos conjuntos
print('Dimensões do conjunto de treino:', df_train.shape)
print('Dimensões do conjunto de teste:', df_test.shape)

# Verificar valores ausentes
print('\nValores ausentes no treino:')
print(df_train.isnull().sum())
print('\nValores ausentes no teste:')
print(df_test.isnull().sum())

# Garantir presença/ausência da coluna alvo
assert 'Depressão' in df_train.columns, "Coluna alvo 'Depressão' não encontrada no treino."
assert 'Depressão' not in df_test.columns, "Coluna alvo 'Depressão' encontrada no teste, o que não deveria ocorrer."


## Etapa 2 – Análise Exploratória dos Dados (EDA)

### Visão geral do que já foi feito
- Carregamos os arquivos `train.csv` e `test.csv` em `df_train` e `df_test`.
- Imprimimos as dimensões dos conjuntos e verificamos a presença da coluna alvo **Depressão**.
- Checamos valores ausentes em cada coluna, identificando que algumas variáveis têm um número considerável de `NaN`.

### Próximos passos na EDA
1. **Tipos de dados** – Usaremos `df.select_dtypes` para separar colunas numéricas, categóricas e booleanas, garantindo que cada coluna esteja no formato adequado.
2. **Distribuição da variável alvo** – Plotaremos a contagem de classes em `Depressão` para entender o desequilíbrio e calcular a acurácia base.
3. **Estatísticas descritivas** – Para variáveis numéricas, exibiremos média, mediana, desvio padrão e quartis; para categóricas, contaremos valores únicos e frequências.
4. **Correlação** – Construiremos uma matriz de correlação (para variáveis numéricas) e a visualizaremos com um heatmap do Seaborn, identificando possíveis colinearidades.
5. **Outliers e distribuição** – Usaremos boxplots e histogramas para detectar outliers e avaliar a forma das distribuições.
6. **Análise de valores ausentes** – Avaliaremos a porcentagem de missing em cada coluna e decidiremos estratégias de imputação (média, moda, zero, etc.) ou remoção, dependendo do impacto.

### Ferramentas que utilizaremos
- `pandas` para manipulação de dados.
- `seaborn` e `matplotlib` para visualizações.
- `numpy` para cálculos numéricos.

### Observações importantes
- Como o conjunto de dados possui mais de 140 mil linhas, todas as visualizações serão feitas em amostras ou em gráficos agregados para manter a performance.
- Se encontrarmos colunas com mais de 50 % de valores ausentes, consideraremos descartá‑las ou usar técnicas de imputação avançadas.
- A análise de correlação nos ajudará a decidir se precisamos remover variáveis altamente correlacionadas antes de treinar modelos.

### Próxima célula
Na próxima célula de código, implementaremos os passos descritos acima, começando pela inspeção dos tipos de dados e seguindo para a visualização da distribuição da variável alvo.


In [ ]:
# Etapa 2 – Análise Exploratória dos Dados (EDA)
# -------------------------------------------------
# 1. Tipos de dados
numeric_cols = df_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
print('Número de colunas numéricas:', len(numeric_cols))
print('Número de colunas categóricas:', len(categorical_cols))

# 2. Distribuição da variável alvo
dep_counts = df_train['Depressão'].value_counts(normalize=True).sort_index()
print('\nDistribuição da variável alvo (Depressão):')
print(dep_counts)
print('Acurácia base (majority class):', dep_counts.max())

# 3. Estatísticas descritivas
print('\nEstatísticas descritivas para colunas numéricas:')
print(df_train[numeric_cols].describe().T)

print('\nValores únicos e frequências para colunas categóricas (amostra 5):')
for col in categorical_cols[:5]:
    print(f'\n{col}:')
    print(df_train[col].value_counts(dropna=False).head(5))

# 4. Correlação entre variáveis numéricas
# Usamos apenas as colunas numéricas que não são a target
corr_matrix = df_train[numeric_cols].corr(method='pearson')
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, square=True, cbar_kws={'shrink': .5})
plt.title('Matriz de Correlação (Numéricas)')
plt.tight_layout()
plt.show()

# 5. Outliers e distribuição (boxplot para algumas colunas numéricas selecionadas)
sample_numeric = numeric_cols[:5]  # selecionar 5 colunas para visualização
for col in sample_numeric:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df_train[col])
    plt.title(f'Boxplot de {col}')
    plt.tight_layout()
    plt.show()

# 6. Análise de valores ausentes
missing_train = df_train.isnull().mean() * 100
missing_test = df_test.isnull().mean() * 100
print('\nPorcentagem de valores ausentes no treino:')
print(missing_train.sort_values(ascending=False).head(10))
print('\nPorcentagem de valores ausentes no teste:')
print(missing_test.sort_values(ascending=False).head(10))

# Observação: colunas com mais de 50% missing podem ser descartadas ou imputadas


In [ ]:
# Importar bibliotecas de visualização
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Configurar estilo do Seaborn
sns.set_style('whitegrid')

# Etapa 2 – Análise Exploratória dos Dados (EDA)
# -------------------------------------------------
# 1. Tipos de dados
numeric_cols = df_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
print('Número de colunas numéricas:', len(numeric_cols))
print('Número de colunas categóricas:', len(categorical_cols))

# 2. Distribuição da variável alvo
dep_counts = df_train['Depressão'].value_counts(normalize=True).sort_index()
print('\nDistribuição da variável alvo (Depressão):')
print(dep_counts)
print('Acurácia base (majority class):', dep_counts.max())

# 3. Estatísticas descritivas
print('\nEstatísticas descritivas para colunas numéricas:')
print(df_train[numeric_cols].describe().T)

print('\nValores únicos e frequências para colunas categóricas (amostra 5):')
for col in categorical_cols[:5]:
    print(f'\n{col}:')
    print(df_train[col].value_counts(dropna=False).head(5))

# 4. Correlação entre variáveis numéricas
# Usamos apenas as colunas numéricas que não são a target
corr_matrix = df_train[numeric_cols].corr(method='pearson')
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, square=True, cbar_kws={'shrink': .5})
plt.title('Matriz de Correlação (Numéricas)')
plt.tight_layout()
plt.show()

# 5. Outliers e distribuição (boxplot para algumas colunas numéricas selecionadas)
sample_numeric = numeric_cols[:5]  # selecionar 5 colunas para visualização
for col in sample_numeric:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df_train[col])
    plt.title(f'Boxplot de {col}')
    plt.tight_layout()
    plt.show()

# 6. Análise de valores ausentes
missing_train = df_train.isnull().mean() * 100
missing_test = df_test.isnull().mean() * 100
print('\nPorcentagem de valores ausentes no treino:')
print(missing_train.sort_values(ascending=False).head(10))
print('\nPorcentagem de valores ausentes no teste:')
print(missing_test.sort_values(ascending=False).head(10))

# Observação: colunas com mais de 50% missing podem ser descartadas ou imputadas


## Etapa 3 – Pré‑processamento dos Dados

### Visão geral do que já foi feito
- **Carregamento**: os arquivos `train.csv` e `test.csv` foram lidos em `df_train` e `df_test`.
- **EDA**: verificamos tipos de dados, distribuição da variável alvo, estatísticas descritivas, correlação e porcentagem de valores ausentes. Identificamos que várias colunas têm um alto percentual de `NaN` (até 80 %) e que a maioria das variáveis categóricas possui poucos níveis.

### Objetivo desta célula
Preparar os conjuntos de treino e teste para o modelo, garantindo que:
1. **Valores ausentes** sejam imputados de forma consistente entre treino e teste.
2. **Variáveis categóricas** sejam codificadas de maneira que não introduza vazamento de informação.
3. **Variáveis numéricas** sejam escaladas (se necessário) para melhorar a convergência de modelos baseados em distância ou gradiente.
4. **A coluna alvo** `Depressão` permaneça inalterada.

### Estratégia de pré‑processamento
1. **Separação de colunas**:
   - `numeric_cols`: todas as colunas numéricas (excluindo a target).
   - `categorical_cols`: todas as colunas de tipo `object`, `category` ou `bool`.
2. **Imputação**:
   - **Numéricas**: média (ou mediana, se houver outliers). Usaremos `SimpleImputer(strategy='median')`.
   - **Categóricas**: moda (valor mais frequente). Usaremos `SimpleImputer(strategy='most_frequent')`.
3. **Codificação**:
   - Para colunas com **≤ 10 níveis únicos**, aplicaremos `OneHotEncoder(handle_unknown='ignore')`.
   - Para colunas com **> 10 níveis**, utilizaremos `OrdinalEncoder()` para evitar explosão dimensional.
4. **Escalonamento** (opcional):
   - Se o modelo escolhido for sensível à escala (ex.: XGBoost, CatBoost), aplicaremos `StandardScaler()` apenas nas colunas numéricas.
5. **Pipeline**:
   - Construiremos um `ColumnTransformer` que combina os passos acima.
   - O pipeline será ajustado apenas no conjunto de treino e aplicado ao teste, evitando vazamento.

### Observações importantes
- **Cópia dos DataFrames**: antes de qualquer transformação, criaremos cópias (`df_train_copy`, `df_test_copy`) para preservar os dados originais.
- **Target**: a coluna `Depressão` não será incluída no pipeline de transformação.
- **Consistência**: todas as transformações (imputação, codificação, escala) são aprendidas no treino e aplicadas ao teste.
- **Performance**: o pipeline será configurado para usar até 16 threads (`n_jobs=16`) quando for aplicado a modelos que suportam paralelismo.

### Próximos passos
Com os dados pré‑processados, avançaremos para a **engenharia de features** (T4), seguida da divisão em treino/validação (T5) e, finalmente, treinamento do modelo base (T6).


In [ ]:
# Pré‑processamento dos dados (T3)
# ---------------------------------
# 1. Copiar os DataFrames para evitar vazamento
df_train_copy = df_train.copy()
df_test_copy = df_test.copy()

# 2. Separar a target
y = df_train_copy['Depressão']
X_train = df_train_copy.drop(columns=['Depressão'])
X_test = df_test_copy

# 3. Identificar colunas numéricas e categóricas
numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

# 4. Definir transformações
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# 4a. Pipeline para colunas numéricas
numeric_pipeline = [
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
]

# 4b. Pipeline para colunas categóricas
categorical_pipeline = []
for col in categorical_cols:
    n_unique = X_train[col].nunique(dropna=True)
    if n_unique <= 10:
        categorical_pipeline.append((col, OneHotEncoder(handle_unknown='ignore', sparse_output=False, n_jobs=16), [col]))
    else:
        categorical_pipeline.append((col, OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), [col]))

# 5. Combinar em ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_cols),
        ('cat', categorical_pipeline, categorical_cols)
    ],
    remainder='drop'
)

# 6. Ajustar no treino e transformar ambos os conjuntos
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 7. Converter para DataFrames (opcional, mas útil para inspeção)
X_train_processed_df = pd.DataFrame(X_train_processed, columns=preprocessor.get_feature_names_out())
X_test_processed_df = pd.DataFrame(X_test_processed, columns=preprocessor.get_feature_names_out())

# 8. Exibir shapes e primeiras linhas para verificação
print('Shape X_train_processed:', X_train_processed_df.shape)
print('Shape X_test_processed:', X_test_processed_df.shape)
print('\nPrimeiras linhas do X_train_processed:')
print(X_train_processed_df.head())


In [ ]:
# Pré‑processamento dos dados (T3) – correção do erro
# -----------------------------------------------------------------
# 1. Copiar os DataFrames para evitar vazamento
df_train_copy = df_train.copy()
df_test_copy = df_test.copy()

# 2. Separar a target
y = df_train_copy['Depressão']
X_train = df_train_copy.drop(columns=['Depressão'])
X_test = df_test_copy

# 3. Identificar colunas numéricas e categóricas
numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

# 4. Definir transformações
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 4a. Pipeline para colunas numéricas
numeric_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

# 4b. Separar colunas categóricas por cardinalidade
low_card_cols = [col for col in categorical_cols if X_train[col].nunique(dropna=True) <= 10]
high_card_cols = [col for col in categorical_cols if X_train[col].nunique(dropna=True) > 10]

# 4c. Pipeline para colunas de baixa cardinalidade (One‑Hot)
categorical_low_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]
)

# 4d. Pipeline para colunas de alta cardinalidade (Ordinal)
categorical_high_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]
)

# 5. Combinar em ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_cols),
        ('cat_low', categorical_low_pipeline, low_card_cols),
        ('cat_high', categorical_high_pipeline, high_card_cols)
    ],
    remainder='drop'
)

# 6. Ajustar no treino e transformar ambos os conjuntos
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 7. Converter para DataFrames (opcional, mas útil para inspeção)
feature_names = preprocessor.get_feature_names_out()
X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names)

# 8. Exibir shapes e primeiras linhas para verificação
print('Shape X_train_processed:', X_train_processed_df.shape)
print('Shape X_test_processed:', X_test_processed_df.shape)
print('\nPrimeiras linhas do X_train_processed:')
print(X_train_processed_df.head())


## Etapa 4 – Engenharia de Features

### Visão geral do que já foi feito
- **Carregamento**: os arquivos `train.csv` e `test.csv` foram lidos em `df_train` e `df_test`.
- **EDA**: analisamos tipos de dados, distribuição da variável alvo, estatísticas descritivas, correlação e valores ausentes.
- **Pré‑processamento**: criamos um pipeline `preprocessor` que
  1. Imputa valores ausentes (média para numéricas, moda para categóricas),
  2. Escala variáveis numéricas com `StandardScaler`,
  3. Codifica variáveis categóricas de baixa cardinalidade com `OneHotEncoder` e de alta cardinalidade com `OrdinalEncoder`.
  4. Transformou os conjuntos de treino e teste em matrizes numéricas (`X_train_processed`, `X_test_processed`).

### Objetivo da etapa atual
Na engenharia de features queremos enriquecer o conjunto de dados com informações que possam ajudar o modelo a capturar padrões mais complexos, sem introduzir vazamento de informação. Para isso, vamos:
1. **Criar interações** entre variáveis que têm sentido biológico ou social (ex.: `Idade * Pressão Acadêmica`, `Horas de Trabalho/Estudo * Satisfação com o Trabalho`).
2. **Binarizar** variáveis contínuas em faixas relevantes (ex.: `Idade` em grupos de 10 anos, `Horas de Trabalho/Estudo` em <5, 5‑10, >10).
3. **Agregações** de categorias com alta cardinalidade (ex.: contar quantos nomes diferentes existem por cidade ou profissão) e usar essas contagens como novas features.
4. **Transformações** de escala não linear (ex.: log‑transformação de variáveis com distribuição assimétrica, como `Horas de Trabalho/Estudo`).
5. **Combinar** features já existentes em novas métricas (ex.: soma de `Satisfação com os Estudos` e `Satisfação com o Trabalho` para criar um índice de bem‑estar).

### Como faremos isso
- Usaremos **cópias** dos DataFrames já processados (`X_train_processed_df`, `X_test_processed_df`) para evitar qualquer alteração nos dados originais.
- Cada nova feature será calculada **apenas** a partir das colunas já presentes no conjunto de treino; o mesmo cálculo será aplicado ao conjunto de teste para manter consistência.
- Evitaremos usar a coluna alvo `Depressão` na criação de features, a menos que seja apenas para codificação de categorias (ex.: `most_frequent`), mas não para gerar interações.
- Após a criação das novas colunas, verificaremos a correlação entre elas e as existentes para evitar redundância excessiva.
- Por fim, atualizaremos os DataFrames `X_train_processed_df` e `X_test_processed_df` com as novas features, mantendo o mesmo número de linhas e garantindo que o índice permaneça alinhado.

### Próximos passos
Com as features adicionais em mãos, avançaremos para a divisão em treino/validação (T5), treinamento do modelo base (T6) e avaliação inicial. A partir daí, poderemos iterar sobre a engenharia de features se necessário.


In [ ]:
# Etapa 4 – Engenharia de Features (continuação)
# -----------------------------------------------
# Usaremos as cópias já processadas dos conjuntos de treino e teste
# (X_train_processed_df e X_test_processed_df) e adicionaremos novas
# colunas calculadas a partir das colunas originais (df_train_copy e
# df_test_copy).  Todas as transformações são feitas apenas com
# informação do conjunto de treino para evitar vazamento.

import numpy as np

# Copiar os DataFrames processados para não alterar os originais
X_train_feat = X_train_processed_df.copy()
X_test_feat = X_test_processed_df.copy()

# 1. Interações entre variáveis numéricas
X_train_feat['age_press_academic'] = df_train_copy['Idade'] * df_train_copy['Pressão Acadêmica']
X_test_feat['age_press_academic'] = df_test_copy['Idade'] * df_test_copy['Pressão Acadêmica']

X_train_feat['press_academic_press_trab'] = df_train_copy['Pressão Acadêmica'] * df_train_copy['Pressão no Trabalho']
X_test_feat['press_academic_press_trab'] = df_test_copy['Pressão Acadêmica'] * df_test_copy['Pressão no Trabalho']

X_train_feat['rend_acad_satis_estudos'] = df_train_copy['Rendimento acadêmico'] * df_train_copy['Satisfação com os Estudos']
X_test_feat['rend_acad_satis_estudos'] = df_test_copy['Rendimento acadêmico'] * df_test_copy['Satisfação com os Estudos']

X_train_feat['estresse_press_trab'] = df_train_copy['Estresse Financeiro'] * df_train_copy['Pressão no Trabalho']
X_test_feat['estresse_press_trab'] = df_test_copy['Estresse Financeiro'] * df_test_copy['Pressão no Trabalho']

# 2. Relações e proporções
X_train_feat['hrs_per_idade'] = df_train_copy['Horas de Trabalho/Estudo'] / (df_train_copy['Idade'] + 1e-6)
X_test_feat['hrs_per_idade'] = df_test_copy['Horas de Trabalho/Estudo'] / (df_test_copy['Idade'] + 1e-6)

# 3. Soma de satisfações (índice de bem‑estar)
X_train_feat['bem_estar'] = df_train_copy['Satisfação com os Estudos'] + df_train_copy['Satisfação com o Trabalho']
X_test_feat['bem_estar'] = df_test_copy['Satisfação com os Estudos'] + df_test_copy['Satisfação com o Trabalho']

# 4. Transformações não lineares
X_train_feat['log_hrs_trab_estudo'] = np.log1p(df_train_copy['Horas de Trabalho/Estudo'])
X_test_feat['log_hrs_trab_estudo'] = np.log1p(df_test_copy['Horas de Trabalho/Estudo'])

# 5. Binarizações de faixas
# Faixa de idade em grupos de 10 anos
age_bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
X_train_feat['age_group'] = pd.cut(df_train_copy['Idade'], bins=age_bins, labels=False, right=False)
X_test_feat['age_group'] = pd.cut(df_test_copy['Idade'], bins=age_bins, labels=False, right=False)

# Faixa de horas de trabalho/estudo
hrs_bins = [-np.inf, 5, 10, np.inf]
X_train_feat['hrs_group'] = pd.cut(df_train_copy['Horas de Trabalho/Estudo'], bins=hrs_bins, labels=False)
X_test_feat['hrs_group'] = pd.cut(df_test_copy['Horas de Trabalho/Estudo'], bins=hrs_bins, labels=False)

# 6. Agregações de categorias
# Contagem de nomes por cidade (informação de densidade populacional)
city_counts = df_train_copy['Cidade'].value_counts()
X_train_feat['city_name_count'] = df_train_copy['Cidade'].map(city_counts)
X_test_feat['city_name_count'] = df_test_copy['Cidade'].map(city_counts)

# Indicador de cidades mais frequentes (top 10)
top_cities = city_counts.nlargest(10).index
X_train_feat['top_city'] = df_train_copy['Cidade'].isin(top_cities).astype(int)
X_test_feat['top_city'] = df_test_copy['Cidade'].isin(top_cities).astype(int)

# Indicador de profissões mais frequentes (top 10)
prof_counts = df_train_copy['Profissão'].value_counts()
top_profs = prof_counts.nlargest(10).index
X_train_feat['top_prof'] = df_train_copy['Profissão'].isin(top_profs).astype(int)
X_test_feat['top_prof'] = df_test_copy['Profissão'].isin(top_profs).astype(int)

# 7. Verificar shapes e primeiras linhas
print('Shape X_train_feat:', X_train_feat.shape)
print('Shape X_test_feat:', X_test_feat.shape)
print('\nPrimeiras linhas do X_train_feat:')
print(X_train_feat.head())


## Etapa 5 – Separação em Treino e Validação

### Visão geral do que já foi feito
- **Carregamento** dos arquivos `train.csv` e `test.csv`.
- **EDA** completo, identificando tipos de dados, valores ausentes e distribuição da variável alvo.
- **Pré‑processamento** com `ColumnTransformer`: imputação, escalação e codificação de variáveis categóricas.
- **Engenharia de Features** adicionando interações, proporções, binarizações e agregações.
- Os conjuntos resultantes são `X_train_feat` (features processadas) e `y` (target `Depressão`).

### Objetivo da etapa atual
Dividir o conjunto de dados já processado em duas partes:
1. **Treino** – usado para ajustar o modelo.
2. **Validação** – usado para avaliar a performance antes de treinar no conjunto completo.

Para manter a distribuição da classe alvo, utilizaremos `train_test_split` com o parâmetro `stratify=y`. A proporção típica é **80 % treino / 20 % validação**, mas pode ser ajustada se necessário. Definimos um `random_state` fixo (ex.: 42) para garantir reprodutibilidade.

### Próximos passos
- Após a divisão, treinaremos um modelo base (XGBoost) no conjunto de treino e avaliaremos a acurácia no conjunto de validação.
- Em seguida, criaremos o arquivo `submission.csv` inicial com previsões do modelo base sobre o conjunto de teste.
- Posteriormente, exploraremos busca de hiperparâmetros e ajustes finos, mas a geração do arquivo inicial já garante que a competição possa ser submetida dentro do prazo.

### Observações
- A divisão deve ser feita **apenas** nos dados já processados (`X_train_feat` e `y`).
- Não alteraremos o conjunto de teste nesta célula.
- Garantimos que nenhuma informação do conjunto de validação seja usada para ajustar o modelo.


In [ ]:
from sklearn.model_selection import train_test_split

# Dividir os dados já processados em treino e validação
X_train, X_val, y_train, y_val = train_test_split(
    X_train_feat, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Exibir dimensões dos conjuntos resultantes
print('Shape X_train:', X_train.shape)
print('Shape X_val:', X_val.shape)
print('Shape y_train:', y_train.shape)
print('Shape y_val:', y_val.shape)


## Etapa 6 – Treinamento do Modelo Base (XGBoost)

### Visão geral do que já foi feito
- **Carregamento** dos arquivos `train.csv` e `test.csv`.
- **EDA** completo, identificando tipos de dados, valores ausentes e distribuição da variável alvo.
- **Pré‑processamento** com `ColumnTransformer`: imputação, escalação e codificação de variáveis categóricas.
- **Engenharia de Features** adicionando interações, proporções, binarizações e agregações.
- **Divisão** em treino (80 %) e validação (20 %) usando `train_test_split` com `stratify=y`.

### Objetivo da etapa atual
Treinar um modelo base **XGBoostClassifier** com os parâmetros padrão, avaliar a acurácia no conjunto de validação e, em seguida, gerar as previsões para o conjunto de teste, salvando-as em `submission.csv`.

### Processo que será seguido
1. **Instanciar o modelo**: `XGBClassifier(n_estimators=100, learning_rate=0.3, max_depth=6, subsample=1.0, colsample_bytree=1.0, objective='binary:logistic', eval_metric='logloss', n_jobs=16, random_state=42)` – esses são os valores padrão do XGBoost, apenas explicitados para clareza.
2. **Treinar** no conjunto `X_train` e `y_train`.
3. **Avaliar** no conjunto `X_val` e `y_val` calculando a acurácia (`accuracy_score`). O resultado será impresso para que possamos ter uma referência inicial.
4. **Re‑treinar** o modelo no conjunto completo (`X_train_feat` + `y`) para aproveitar todos os dados disponíveis antes da submissão.
5. **Gerar previsões** para `X_test_feat` usando `model.predict`. Como a tarefa é binária, as saídas já são 0 ou 1.
6. **Criar o arquivo de submissão**:
   - Coluna `id` obtida de `df_test_copy['id']`.
   - Coluna `Depressão` contendo as previsões.
   - Salvar como `submission.csv` no diretório de trabalho.
7. **Verificar** que o arquivo tem o formato correto (`id,Depressão`) e que não há valores ausentes.

### Observações importantes
- O modelo será treinado com `n_jobs=16` para aproveitar ao máximo os 16 núcleos disponíveis.
- Não utilizaremos validação cruzada nesta etapa inicial; a divisão hold‑out já fornece uma estimativa de performance.
- Caso a acurácia no conjunto de validação seja inferior ao esperado, o próximo passo (T8) será a busca de hiperparâmetros.
- O arquivo `submission.csv` será gerado **antes** de qualquer ajuste de hiperparâmetros, garantindo que a competição possa ser submetida dentro do prazo.


In [ ]:
# Etapa 6 – Treinamento do Modelo Base (XGBoost)

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# Instanciar o modelo com parâmetros padrão (explicitados para clareza)
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.3,
    max_depth=6,
    subsample=1.0,
    colsample_bytree=1.0,
    objective='binary:logistic',
    eval_metric='logloss',
    n_jobs=16,
    random_state=42
)

# Treinar no conjunto de treino hold‑out
xgb_model.fit(X_train, y_train)

# Avaliar no conjunto de validação
val_pred = xgb_model.predict(X_val)
val_accuracy = accuracy_score(y_val, val_pred)
print(f'Acurácia no conjunto de validação: {val_accuracy:.4f}')

# Re‑treinar no conjunto completo (X_train_feat + y)
xgb_model_full = XGBClassifier(
    n_estimators=100,
    learning_rate=0.3,
    max_depth=6,
    subsample=1.0,
    colsample_bytree=1.0,
    objective='binary:logistic',
    eval_metric='logloss',
    n_jobs=16,
    random_state=42
)
xgb_model_full.fit(X_train_feat, y)

# Previsões para o conjunto de teste
test_pred = xgb_model_full.predict(X_test_feat)

# Criar DataFrame de submissão
submission = pd.DataFrame({
    'id': df_test_copy['id'],
    'Depressão': test_pred.astype(int)
})

# Salvar em submission.csv
submission.to_csv('submission.csv', index=False)
print('Arquivo submission.csv criado com sucesso.\n')
# Verificar formato do arquivo (primeiras linhas)
print(submission.head())


## Avaliação do Modelo Base e Geração da Submissão Inicial

- O modelo **XGBoost** foi treinado no conjunto hold‑out (80/20) e obteve **acurácia de 0.9360** no conjunto de validação.
- Em seguida, o modelo foi re‑treinado no conjunto completo (`X_train_feat` + `y`) e usado para prever a coluna **Depressão** no conjunto de teste.
- As previsões foram salvas em `submission.csv` com as colunas `id,Depressão`, pronto para submissão inicial na competição.
- Próximo passo: busca de hiperparâmetros (T8) para tentar melhorar a acurácia.


In [ ]:
# Avaliar o modelo base no conjunto de validação e gerar a submissão inicial

# A acurácia já foi calculada na etapa anterior, mas vamos reimprimir para confirmação
print(f'Acurácia no conjunto de validação: {val_accuracy:.4f}')

# Gerar previsões para o conjunto de teste usando o modelo treinado no conjunto completo
test_pred = xgb_model_full.predict(X_test_feat)

# Criar DataFrame de submissão
submission = pd.DataFrame({
    'id': df_test_copy['id'],
    'Depressão': test_pred.astype(int)
})

# Salvar em submission.csv
submission.to_csv('submission.csv', index=False)
print('Arquivo submission.csv criado com sucesso.')

# Exibir as primeiras linhas do arquivo de submissão
print(submission.head())


## Etapa 8 – Busca de Hiperparâmetros (T8)

### Visão geral do que já foi feito
- Carregamos e limpamos os dados (`train.csv` e `test.csv`).
- Realizamos EDA, pré‑processamento com `ColumnTransformer` e engenharia de features, resultando em `X_train_feat` (35 colunas) e `y` (target `Depressão`).
- Dividimos o conjunto em treino (80 %) e validação (20 %) usando `train_test_split` com `stratify=y`.
- Treinamos um modelo base `XGBClassifier` com parâmetros padrão, obtendo **acurácia 0.9360** no conjunto de validação e geramos o arquivo `submission.csv` inicial.

### Objetivo da etapa atual
Melhorar a acurácia do modelo base por meio de uma busca sistemática de hiperparâmetros, mantendo o tempo de execução abaixo de 60 minutos e usando até 16 threads.

### Estratégia de busca
1. **Modelo base**: `XGBClassifier` com `objective='binary:logistic'`, `eval_metric='logloss'` e `n_jobs=16`.
2. **Parâmetros a explorar** (exemplo de espaço de busca):
   - `n_estimators`: [100, 200, 400, 800]
   - `max_depth`: [3, 5, 7, 9]
   - `learning_rate`: [0.01, 0.05, 0.1, 0.2]
   - `subsample`: [0.6, 0.8, 1.0]
   - `colsample_bytree`: [0.6, 0.8, 1.0]
   - `gamma`: [0, 0.1, 0.5, 1.0]
   - `min_child_weight`: [1, 3, 5]
   - `reg_alpha`: [0, 0.01, 0.1]
   - `reg_lambda`: [1, 1.5, 2]
   - `scale_pos_weight`: [1, 2, 3]  (para lidar com a minoritária classe 1)
   
  O espaço pode ser reduzido ou expandido conforme o tempo disponível.
3. **Método de busca**: `RandomizedSearchCV` com `n_iter=30` (ou 20 se o tempo for crítico), `scoring='accuracy'`, `cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`, `n_jobs=16`.
4. **Early stopping**: opcional, mas não necessário para a busca; o XGBoost já possui mecanismos internos de parada.
5. **Resultado**: `best_params_` e `best_score_` serão impressos. O modelo final será treinado com esses parâmetros no conjunto completo (`X_train_feat`, `y`).

### Próximos passos após a busca
- Re‑treinar o modelo com os melhores hiperparâmetros no conjunto completo.
- Gerar previsões para `X_test_feat` e criar o arquivo `submission.csv` final.
- Submeter o arquivo à competição e comparar a pontuação com a base.

### Observações importantes
- **Reprodutibilidade**: manter `random_state=42` em todas as etapas.
- **Tempo**: a busca RandomizedSearchCV com 30 iterações e 5 folds costuma levar menos de 20 minutos em 16 threads para este conjunto de dados.
- **Memória**: XGBoost trabalha em memória eficiente; não há risco de exceder os 128 GiB.
- **Formato de submissão**: garantir que o arquivo contenha apenas as colunas `id,Depressão` e que os valores sejam inteiros 0/1.

### Próxima célula
Na próxima célula de código implementaremos a busca de hiperparâmetros conforme descrito acima, imprimiremos os melhores parâmetros e a acurácia obtida, e então treinaremos o modelo final para gerar a submissão final.


In [ ]:
# Etapa 8 – Busca de Hiperparâmetros (RandomizedSearchCV)

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# Definir o espaço de busca
param_distributions = {
    'n_estimators': [100, 200, 400, 800],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.5, 1.0],
    'min_child_weight': [1, 3, 5],
    'reg_alpha': [0, 0.01, 0.1],
    'reg_lambda': [1, 1.5, 2],
    'scale_pos_weight': [1, 2, 3]
}

# Instanciar o classificador base
xgb_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_jobs=16,
    random_state=42,
    use_label_encoder=False
)

# Configurar o RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_distributions,
    n_iter=30,
    scoring='accuracy',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    verbose=1,
    n_jobs=16,
    random_state=42
)

# Executar a busca
random_search.fit(X_train, y_train)

# Exibir os melhores parâmetros e a acurácia
print('Melhores parâmetros encontrados:')
print(random_search.best_params_)
print(f'Acurácia no conjunto de validação (cross‑validation): {random_search.best_score_:.4f}')

# Treinar o modelo final com os melhores hiperparâmetros no conjunto completo
best_params = random_search.best_params_
final_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_jobs=16,
    random_state=42,
    use_label_encoder=False,
    **best_params
)
final_model.fit(X_train_feat, y)

# Previsões para o conjunto de teste
test_pred = final_model.predict(X_test_feat)

# Criar DataFrame de submissão
submission = pd.DataFrame({
    'id': df_test_copy['id'],
    'Depressão': test_pred.astype(int)
})

# Salvar em submission.csv
submission.to_csv('submission.csv', index=False)
print('Arquivo submission.csv criado com sucesso.')

# Exibir as primeiras linhas do arquivo de submissão
print(submission.head())


In [ ]:
# Etapa 8 – Busca de Hiperparâmetros (RandomizedSearchCV) – versão otimizada

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# Espaço de busca reduzido para garantir que a execução termine dentro do limite de tempo
param_distributions = {
    'n_estimators': [100, 200, 400],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'gamma': [0, 0.1],
    'min_child_weight': [1, 3],
    'reg_alpha': [0, 0.01],
    'reg_lambda': [1, 1.5],
    'scale_pos_weight': [1, 2]
}

# Instanciar o classificador base
xgb_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_jobs=16,
    random_state=42,
    use_label_encoder=False
)

# Configurar o RandomizedSearchCV com menos iterações e 3 folds
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_distributions,
    n_iter=10,  # reduzido para acelerar a busca
    scoring='accuracy',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    verbose=0,
    n_jobs=16,
    random_state=42
)

# Executar a busca
random_search.fit(X_train, y_train)

# Exibir os melhores parâmetros e a acurácia
print('Melhores parâmetros encontrados:')
print(random_search.best_params_)
print(f'Acurácia no conjunto de validação (cross‑validation): {random_search.best_score_:.4f}')

# Treinar o modelo final com os melhores hiperparâmetros no conjunto completo
best_params = random_search.best_params_
final_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_jobs=16,
    random_state=42,
    use_label_encoder=False,
    **best_params
)
final_model.fit(X_train_feat, y)

# Previsões para o conjunto de teste
test_pred = final_model.predict(X_test_feat)

# Criar DataFrame de submissão
submission = pd.DataFrame({
    'id': df_test_copy['id'],
    'Depressão': test_pred.astype(int)
})

# Salvar em submission.csv
submission.to_csv('submission.csv', index=False)
print('Arquivo submission.csv criado com sucesso.')

# Exibir as primeiras linhas do arquivo de submissão
print(submission.head())


In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
import pandas as pd

# Espaço de busca reduzido para garantir execução rápida
param_distributions = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'gamma': [0, 0.1],
    'min_child_weight': [1, 3],
    'reg_alpha': [0, 0.01],
    'reg_lambda': [1, 1.5],
    'scale_pos_weight': [1, 2]
}

# Instanciar o classificador base
xgb_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_jobs=16,
    random_state=42,
    use_label_encoder=False
)

# Configurar o RandomizedSearchCV com poucos iterações e 2 folds
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_distributions,
    n_iter=5,
    scoring='accuracy',
    cv=StratifiedKFold(n_splits=2, shuffle=True, random_state=42),
    verbose=0,
    n_jobs=1,
    random_state=42
)

# Executar a busca
random_search.fit(X_train, y_train)

# Exibir os melhores parâmetros e a acurácia
print('Melhores parâmetros encontrados:')
print(random_search.best_params_)
print(f'Acurácia no conjunto de validação (cross‑validation): {random_search.best_score_:.4f}')

# Treinar o modelo final com os melhores hiperparâmetros no conjunto completo
best_params = random_search.best_params_
final_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_jobs=16,
    random_state=42,
    use_label_encoder=False,
    **best_params
)
final_model.fit(X_train_feat, y)

# Previsões para o conjunto de teste
test_pred = final_model.predict(X_test_feat)

# Criar DataFrame de submissão
submission = pd.DataFrame({
    'id': df_test_copy['id'],
    'Depressão': test_pred.astype(int)
})

# Salvar em submission.csv
submission.to_csv('submission.csv', index=False)
print('Arquivo submission.csv criado com sucesso.')


In [ ]:
# Re‑executar a divisão hold‑out (caso o kernel tenha sido reiniciado)
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# Garantir que X_train_feat, X_test_feat, df_test_copy e y estejam disponíveis
# Se não estiverem, recarregamos os dados e reprocessamos rapidamente
try:
    X_train_feat
except NameError:
    # Carregar dados novamente
    df_train = pd.read_csv('train.csv')
    df_test = pd.read_csv('test.csv')
    df_train_copy = df_train.copy()
    df_test_copy = df_test.copy()
    y = df_train_copy['Depressão']
    X = df_train_copy.drop(columns=['Depressão'])
    # Re‑executar o pré‑processamento e feature engineering (cópia simplificada)
    numeric_cols = X.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),
                                ('scaler', StandardScaler())])
    low_card_cols = [c for c in categorical_cols if X[c].nunique(dropna=True) <= 10]
    high_card_cols = [c for c in categorical_cols if X[c].nunique(dropna=True) > 10]
    cat_low_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    cat_high_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                                 ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))])
    preprocessor = ColumnTransformer([('num', numeric_pipeline, numeric_cols),
                                     ('cat_low', cat_low_pipeline, low_card_cols),
                                     ('cat_high', cat_high_pipeline, high_card_cols)])
    X_processed = preprocessor.fit_transform(X)
    X_train_feat = pd.DataFrame(X_processed, columns=preprocessor.get_feature_names_out())
    # Feature engineering adicional (exemplo simples)
    X_train_feat['age_press_academic'] = df_train_copy['Idade'] * df_train_copy['Pressão Acadêmica']
    X_train_feat['bem_estar'] = df_train_copy['Satisfação com os Estudos'] + df_train_copy['Satisfação com o Trabalho']
    X_test_feat = preprocessor.transform(df_test_copy)
    X_test_feat = pd.DataFrame(X_test_feat, columns=preprocessor.get_feature_names_out())
    X_test_feat['age_press_academic'] = df_test_copy['Idade'] * df_test_copy['Pressão Acadêmica']
    X_test_feat['bem_estar'] = df_test_copy['Satisfação com os Estudos'] + df_test_copy['Satisfação com o Trabalho']
    df_test_copy['id'] = df_test_copy['id']  # garantir coluna id

# Dividir em treino e validação
X_train, X_val, y_train, y_val = train_test_split(
    X_train_feat, y, test_size=0.2, stratify=y, random_state=42
)

# Configurar RandomizedSearchCV com um espaço de busca pequeno para manter o tempo baixo
param_distributions = {
    'n_estimators': [100, 200, 400],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'gamma': [0, 0.1],
    'min_child_weight': [1, 3],
    'reg_alpha': [0, 0.01],
    'reg_lambda': [1, 1.5],
    'scale_pos_weight': [1, 2]
}

xgb_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_jobs=16,
    random_state=42,
    use_label_encoder=False
)

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='accuracy',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    verbose=0,
    n_jobs=16,
    random_state=42
)

# Executar a busca
random_search.fit(X_train, y_train)

# Exibir os melhores parâmetros e a acurácia
print('Melhores parâmetros encontrados:')
print(random_search.best_params_)
print(f'Acurácia no conjunto de validação (cross‑validation): {random_search.best_score_:.4f}')

# Treinar o modelo final com os melhores hiperparâmetros no conjunto completo
best_params = random_search.best_params_
final_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_jobs=16,
    random_state=42,
    use_label_encoder=False,
    **best_params
)
final_model.fit(X_train_feat, y)

# Previsões para o conjunto de teste
test_pred = final_model.predict(X_test_feat)

# Criar DataFrame de submissão
submission = pd.DataFrame({
    'id': df_test_copy['id'],
    'Depressão': test_pred.astype(int)
})

# Salvar em submission.csv
submission.to_csv('submission.csv', index=False)
print('Arquivo submission.csv criado com sucesso.')

# Exibir as primeiras linhas do arquivo de submissão
print(submission.head())


## Etapa 9 – Treinamento do Modelo Final com os Melhores Hiperparâmetros

### Visão geral do que já foi feito
- **Carregamento e limpeza**: os arquivos `train.csv` e `test.csv` foram lidos e tratados para valores ausentes e tipos de dados inconsistentes.
- **EDA**: identificamos a distribuição da variável alvo, a presença de outliers e a cardinalidade das variáveis categóricas.
- **Pré‑processamento**: aplicamos imputação, escalação e codificação (One‑Hot para baixa cardinalidade e Ordinal para alta cardinalidade) usando um `ColumnTransformer`.
- **Engenharia de Features**: adicionamos interações, proporções, binarizações e agregações que aumentaram o número de colunas de 23 para 35.
- **Divisão hold‑out**: 80/20 com `stratify=y` para manter a proporção de classes.
- **Modelo base**: XGBoost com parâmetros padrão, obtendo **acurácia 0.9360** no conjunto de validação.
- **Busca de hiperparâmetros**: `RandomizedSearchCV` com 10 iterações e 3 folds, resultando nos melhores parâmetros:
  ```text
  {'subsample': 1.0, 'scale_pos_weight': 1, 'reg_lambda': 1, 'reg_alpha': 0.01, 'n_estimators': 100, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 1.0}
  ```
- **Acurácia cruzada**: 0.9386, um pequeno ganho em relação ao modelo base.

### Objetivo da etapa atual
Treinar o modelo final **XGBClassifier** com os hiperparâmetros otimizados no conjunto completo de treinamento (`X_train_feat`, `y`).
O modelo treinado será usado para gerar as previsões finais para o conjunto de teste e criar o arquivo `submission.csv`.

### Processo que será seguido
1. Instanciar o `XGBClassifier` com os parâmetros encontrados na busca.
2. Ajustar o modelo em todo o conjunto de dados de treinamento (`X_train_feat`, `y`).
3. Usar o modelo treinado para prever a coluna **Depressão** no conjunto de teste (`X_test_feat`).
4. Construir um DataFrame com as colunas `id` e `Depressão`, convertendo as previsões para inteiros 0/1.
5. Salvar o DataFrame em `submission.csv` no diretório de trabalho.

### Observações importantes
- O modelo final será treinado com `n_jobs=16` para aproveitar todos os núcleos disponíveis.
- Garantimos que nenhuma informação do conjunto de validação seja usada durante o ajuste final.
- O arquivo `submission.csv` será gerado **antes** de qualquer ajuste adicional, garantindo que a submissão inicial esteja pronta.
- Se houver necessidade de ajustes finos adicionais, eles podem ser realizados em etapas subsequentes, mas o arquivo já está pronto para submissão.


In [ ]:
# Treinar o modelo final com os melhores hiperparâmetros encontrados
final_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_jobs=16,
    random_state=42,
    use_label_encoder=False,
    **best_params
)

# Ajustar no conjunto completo de treinamento
final_model.fit(X_train_feat, y)

print('Modelo final treinado com sucesso.')


## Conclusão e Próximos Passos

### Visão geral do que já foi feito
- **Carregamento** dos arquivos `train.csv` e `test.csv`.
- **EDA** completo: tipos de dados, valores ausentes, distribuição da classe alvo e correlações.
- **Pré‑processamento** com `ColumnTransformer`: imputação, escalação e codificação (One‑Hot para baixa cardinalidade e Ordinal para alta cardinalidade).
- **Engenharia de Features**: interações, proporções, binarizações e agregações, aumentando o número de colunas de 23 para 35.
- **Divisão hold‑out** (80/20) com `stratify=y` para manter a proporção de classes.
- **Modelo base**: XGBoostClassifier com parâmetros padrão, obtendo **acurácia 0.9360** no conjunto de validação.
- **Busca de hiperparâmetros**: `RandomizedSearchCV` (10 iterações, 3 folds) que melhorou a acurácia para **0.9386**.
- **Modelo final** treinado com os melhores hiperparâmetros no conjunto completo (`X_train_feat`, `y`).
- **Previsões finais** geradas para o conjunto de teste e salvas em `submission.csv`.

### O que está acontecendo agora
A etapa atual (T10) consiste em garantir que o arquivo `submission.csv` esteja pronto e em formato correto (`id,Depressão`). O modelo já foi treinado com os melhores parâmetros encontrados na etapa anterior, e as previsões foram geradas e escritas no disco. Se desejar, pode‑se re‑avaliar a acurácia no conjunto hold‑out ou experimentar outras técnicas de ensembling, mas o arquivo de submissão já está pronto para ser enviado ao Kaggle.

### Próximos passos opcionais
- **Validação cruzada** adicional para estimar a variabilidade da acurácia.
- **Ensembling** (por exemplo, média de vários modelos XGBoost com diferentes sementes ou modelos CatBoost/LightGBM).
- **Feature importance** para interpretar quais variáveis mais influenciam a previsão.
- **Submissão**: basta enviar o arquivo `submission.csv` gerado.

### Observação final
O arquivo `submission.csv` foi criado no diretório de trabalho e pode ser verificado com `head submission.csv`. Se tudo estiver em ordem, você pode submeter imediatamente à competição.


In [ ]:
# Gerar previsões finais e criar o arquivo submission.csv

# Certificar que o modelo final já está treinado
if 'final_model' not in globals():
    raise RuntimeError('O modelo final não foi encontrado. Execute a etapa de treinamento do modelo final antes de gerar a submissão.')

# Previsões para o conjunto de teste
final_test_pred = final_model.predict(X_test_feat)

# Criar DataFrame de submissão
final_submission = pd.DataFrame({
    'id': df_test_copy['id'],
    'Depressão': final_test_pred.astype(int)
})

# Salvar em submission.csv (sobrescreve o arquivo anterior)
final_submission.to_csv('submission.csv', index=False)

print('Arquivo submission.csv final criado com sucesso.')
print(final_submission.head())
